# Our own DeBERTa member — a frozen-encoder probe

Five full fine-tune attempts at DeBERTa-v3-large all diverged: fp32 went NaN at step ~200, bf16 autocast went
NaN on the *first* step, and after removing autocast, lowering the learning rate to 5e-6/5e-5 and adding loss
and gradient guards it still produced a non-finite loss on every batch after the first optimizer step. Chasing
that further would spend the rest of the week's quota on one architecture.

The checkpoint we already load is **a frozen encoder plus a linear head on pooled features**. So the honest
version of "train our own member" is to keep the encoder frozen and fit that head on 98k pool passages:

* **deterministic** — a bounded logistic fit, nothing can diverge;
* **fast** — features extracted once, head fitted in seconds instead of two GPU-hours;
* **drop-in** — byte-for-byte the same `encoder/` + `head.safetensors` + `meta.json` contract.

What it can buy: the frozen encoder's own head scores **0.523** balanced accuracy on unseen prompts and was
trained on only 2,699 rows. Refitting the readout on 98k clean passages should beat that. It will **not** reach
what a full fine-tune would, and this notebook says so rather than pretending otherwise.

The scaler is folded into the saved head (`W' = W/std`, `b' = b - (mean/std)@W.T`) and the fold is asserted
against the probe's own predictions, so the checkpoint stays plain `{weight, bias}`.

In [ ]:
"""Train our DeBERTa member as a linear probe on the frozen encoder — not a full fine-tune.

Five attempts at fine-tuning DeBERTa-v3-large all diverged: fp32 went NaN at step ~200, bf16 autocast went
NaN on the first step, and after removing autocast and lowering the learning rate it still produced a
non-finite loss on every batch after the first optimizer step. Chasing that further would spend the rest of
the week's quota on one architecture.

The published checkpoint we already load is a *frozen encoder plus a linear head on pooled features* —
`encoder/`, `head.safetensors`, `meta.json`. So the honest version of "train our own member" here is to keep
the encoder frozen and fit that head on 98k pool passages. That is:

  * **deterministic** — one closed-form-ish weighted least-squares / logistic fit, no divergence possible;
  * **fast** — features are extracted once, then the head trains in seconds instead of two GPU-hours;
  * **drop-in** — byte-for-byte the same checkpoint contract the inference cell reads.

What it can and cannot buy: the frozen encoder's own head scores 0.523 balanced accuracy on unseen prompts.
A head refit on 98k clean passages should beat that, because the frozen features already separate most of the
18 models and only the readout was trained on 2,699 rows. It will not reach what a full fine-tune would, and
we say so rather than pretending.
"""
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch

T0 = time.time()
INPUT_DIR = "/kaggle/input"
OUT = "/kaggle/working/deberta_ours"
os.makedirs(OUT, exist_ok=True)

CFG = {
    "max_len": 192,
    "bs": 64,
    "pool_cap_per_class": 6000,
    "val_frac": 0.15,
    "seed": 7,
    "C": 1.0,               # inverse L2 strength for the logistic head
    "max_iter": 400,
}
print("CONFIG:", json.dumps(CFG), flush=True)


def find(pat):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pat}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"attach {pat}")
    return hits[0]


DATA = next(p for p in [f"{INPUT_DIR}/competitions/name-that-llm", f"{INPUT_DIR}/name-that-llm"]
            if os.path.exists(f"{p}/train.csv"))
train = pd.read_csv(f"{DATA}/train.csv")
LABELS = sorted(train.model.unique().tolist())
K = len(LABELS)
LAB2I = {m: i for i, m in enumerate(LABELS)}
print(f"official {len(train):,} rows, {K} labels", flush=True)

## Pool: every parquet with text+model, capped per class; prompt-grouped holdout

In [ ]:
"""Every parquet with text+model columns, capped per class."""
frames, seen = [], set()
for p in sorted(glob.glob(f"{INPUT_DIR}/**/*.parquet", recursive=True)):
    if p in seen or "name-that-llm" in p:
        continue
    seen.add(p)
    try:
        head = pd.read_parquet(p).head(3)
    except Exception:
        continue
    cols = {c.lower(): c for c in head.columns}
    t, m = cols.get("text"), (cols.get("model") or cols.get("label"))
    if t is None or m is None:
        continue
    try:
        d = pd.read_parquet(p, columns=[t, m]).rename(columns={t: "text", m: "model"})
    except Exception:
        continue
    if not d.model.isin(LABELS).any():
        continue
    frames.append(d[d.model.isin(LABELS)])
    print(f"  {p.split('/')[-1][:40]:<40} {len(d):>9,}", flush=True)

pool = pd.concat(frames, ignore_index=True)
pool["text"] = pool.text.astype(str)
pool = pool[pool.text.str.split().str.len().between(30, 300)]
pool = pool.drop_duplicates(subset=["text"])
print(f"pool after filter/dedup: {len(pool):,}", flush=True)

pool = pd.concat([g.sample(min(len(g), CFG["pool_cap_per_class"]), random_state=CFG["seed"])
                  for _, g in pool.groupby("model")], ignore_index=True)
print(f"pool used: {len(pool):,}, per class {pool.model.value_counts().min()}-{pool.model.value_counts().max()}",
      flush=True)

prompts = np.array(sorted(train.prompt_id.unique()))
np.random.default_rng(CFG["seed"]).shuffle(prompts)
n_val = max(1, int(len(prompts) * CFG["val_frac"]))
val_p = set(prompts[:n_val].tolist())
va = train[train.prompt_id.isin(val_p)].reset_index(drop=True)
tr_off = train[~train.prompt_id.isin(val_p)].reset_index(drop=True)
print(f"val {len(va):,} rows / {va.prompt_id.nunique()} prompts | official-train {len(tr_off):,} folded in",
      flush=True)

Xdf = pd.concat([pool[["text", "model"]], tr_off[["text", "model"]]], ignore_index=True)
print(f"feature rows: {len(Xdf):,}", flush=True)

## Extract pooled features once — no gradients, so nothing can diverge

In [ ]:
"""Extract pooled features once — the encoder never sees a gradient, so nothing can diverge."""
from transformers import AutoConfig, AutoModel, AutoTokenizer

CKPT = os.path.dirname(find("deberta_v3_large/meta.json"))
enc_dir = os.path.join(CKPT, "encoder")
meta_src = json.load(open(os.path.join(CKPT, "meta.json")))
print("encoder from", enc_dir, "| source meta keys:", list(meta_src)[:6], flush=True)

tok = AutoTokenizer.from_pretrained(enc_dir)
enc = AutoModel.from_pretrained(enc_dir, dtype=torch.float32).to("cuda:0").eval()
H = enc.config.hidden_size
print(f"hidden {H}, max_len {CFG['max_len']}", flush=True)


@torch.inference_mode()
def feats(texts, tag=""):
    out = np.zeros((len(texts), H), np.float32)
    order = np.argsort([len(t.split()) for t in texts], kind="stable")   # sort by length: less padding
    t0 = time.time()
    for s in range(0, len(order), CFG["bs"]):
        idx = order[s:s + CFG["bs"]]
        bt = tok([texts[i] for i in idx], truncation=True, max_length=CFG["max_len"],
                 padding=True, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        h = enc(**bt).last_hidden_state
        out[idx] = ((h * m).sum(1) / m.sum(1).clamp(min=1)).float().cpu().numpy()
        if (s // CFG["bs"]) % 200 == 0:
            print(f"  {tag} {s:>7,}/{len(order):,} ({time.time()-t0:.0f}s)", flush=True)
    return out


Z_tr = feats(Xdf.text.tolist(), "train")
y_tr = Xdf.model.map(LAB2I).to_numpy(np.int64)
Z_va = feats(va.text.tolist(), "val")
y_va = va.model.map(LAB2I).to_numpy(np.int64)
print(f"features: train {Z_tr.shape} | val {Z_va.shape} | {time.time()-T0:.0f}s total", flush=True)
np.save("/kaggle/working/Z_tr.npy", Z_tr)
np.save("/kaggle/working/Z_va.npy", Z_va)
torch.cuda.empty_cache()   # the encoder is released with the cell; no `del` so pyflakes does not flag feats()

## Fit the head

In [ ]:
"""Fit the head. Standardise, class-weight, logistic regression — deterministic and bounded."""
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(Z_tr)
Zs = scaler.transform(Z_tr).astype(np.float32)
Zv = scaler.transform(Z_va).astype(np.float32)

clf = LogisticRegression(C=CFG["C"], max_iter=CFG["max_iter"], class_weight="balanced",
                         multi_class="multinomial", n_jobs=-1)
clf.fit(Zs, y_tr)
pred = clf.predict(Zv)
ba = float(balanced_accuracy_score(y_va, pred))
per = [(pred[y_va == c] == c).mean() for c in range(K) if (y_va == c).any()]
print(f"\nPROBE HOLDOUT BA {ba:.4f} on {len(y_va):,} rows / {va.prompt_id.nunique()} prompts", flush=True)
print(f"per-class recall min/median/max: {np.min(per):.3f}/{np.median(per):.3f}/{np.max(per):.3f}", flush=True)
print(f"frozen-encoder reference BA: {meta_src.get('holdout_ba', 'unknown')} "
      f"(published pool DeBERTa: 0.5763)", flush=True)

## Write the drop-in checkpoint

In [ ]:
"""Write the checkpoint in the exact shape the inference cell loads, with the scaler folded into the head.

The inference cell does `pooled_raw @ W.T + b`. We trained on `(pooled_raw - mean) / std`, so the head must be
rewritten as:

    W' = W / std        b' = b - (mean / std) @ W.T

which keeps the checkpoint at plain `{weight, bias}` and needs no change anywhere else.
"""
from safetensors.torch import save_file

W = clf.coef_.astype(np.float32)                       # (K, H)
b = clf.intercept_.astype(np.float32)                  # (K,)
mean, scale = scaler.mean_.astype(np.float32), scaler.scale_.astype(np.float32)

W_folded = W / scale[None, :]
b_folded = b - (mean / scale) @ W.T

# sanity: the folded head must reproduce the probe's own predictions on the raw features
chk = Z_va @ W_folded.T + b_folded
agree = float((chk.argmax(1) == clf.predict(scaler.transform(Z_va))).mean())
print(f"folded-head agreement with the probe: {agree:.4f}", flush=True)
assert agree > 0.999, "the fold is wrong"

save_file({"weight": torch.from_numpy(W_folded), "bias": torch.from_numpy(b_folded)},
          os.path.join(OUT, "head.safetensors"))

# the encoder ships unchanged — this member only replaces the readout
import shutil
shutil.copytree(enc_dir, os.path.join(OUT, "encoder"), dirs_exist_ok=True)
tok.save_pretrained(os.path.join(OUT, "encoder"))

meta = {
    "labels": LABELS,
    "max_len": CFG["max_len"],
    "hidden_size": H,
    "pooling": "attention-masked mean of last_hidden_state; scaler folded into the head",
    "encoder": "frozen — copied unchanged from the attached deberta_v3_large checkpoint",
    "head": "multinomial logistic regression on standardised pooled features, class_weight=balanced",
    "trained_on": {"rows": int(len(Xdf)), "holdout_rows": int(len(va)),
                   "holdout_prompts": int(va.prompt_id.nunique()), "C": CFG["C"]},
    "holdout_ba": round(float(ba), 4),
    "reference": {"frozen_head_ba": meta_src.get("holdout_ba"),
                  "published_pool_deberta": 0.5763,
                  "note": "five full fine-tune attempts diverged; this is a frozen-encoder probe"},
    "seconds": round(time.time() - T0),
}
json.dump(meta, open(os.path.join(OUT, "meta.json"), "w"), indent=2)

print("\n=== deb_ours (probe) ===", flush=True)
print(json.dumps({k: meta[k] for k in ("holdout_ba", "seconds")}, indent=1), flush=True)
print("files:", sorted(os.listdir(OUT)), flush=True)
print("DROP-IN READY", flush=True)